# Etude pour afficher un résumé des logs.
  * Ne fonctionne qu'avec des fichiers bien structurés. Ne fonctionne qu'avec des dates en français (!). 
  * J'ajoute une surveillance très basique du répertoire de départ des fichiers MsSanté

In [ ]:
# Simplifie le fichier de type ls -ltR sur les logs de Glims
# Idée : produire un fichier html  de synthèse.

from datetime import datetime, timedelta
import re
from pprint import  pprint
from pathlib import Path
from IPython.display import display, HTML

In [ ]:
def parse_ls_lr(contenu):
    resultats = {}
    repertoire_courant = None

    for ligne in contenu.splitlines():
        ligne = ligne.strip()

        # Détection d'un répertoire
        if ligne.startswith("./") and ligne.endswith(":"):
            repertoire_courant = ligne[:-1]  # enlever le ":"
            repertoire_courant = repertoire_courant[2:]
            resultats[repertoire_courant] = None
            continue

        # Détection d'un fichier
        if ligne.startswith("-") and repertoire_courant:
            # Exemple de date : "17 mars 18:40"
            match = re.search(r'(\d{1,2}) (\w+) +(\d{1,2}:\d{2})', ligne)
            if match:
                jour, mois, heure = match.groups()

                # convertir mois FR -> numéro
                mois_fr = {
                    "janv": 1, "févr": 2, "mars": 3, "avril": 4,
                    "mai": 5, "juin": 6, "juil": 7, "août": 8,
                    "sept": 9, "oct": 10, "nov": 11, "déc": 12
                }

#                 mois_num = mois_fr.get(mois[:4].lower())   # ??? pourquoi était il noté mois[:4] ...
                mois_num = mois_fr.get(mois.lower())
                
                if mois_num is None:
                    continue

                date_obj = datetime(
                    year= datetime.now().year,
                    month=mois_num,
                    day=int(jour),
                    hour=int(heure.split(":")[0]),
                    minute=int(heure.split(":")[1])
                )

                # garder la plus récente
                if (resultats[repertoire_courant] is None or
                        date_obj > resultats[repertoire_courant]):
                    resultats[repertoire_courant] = date_obj

    return resultats


In [ ]:
with open(Path("//sn1314/mips/tempo_bma/svc_logs.txt"), 'r') as f:
# with open("logs/svc_logs.txt", 'r') as f:
        lines = f.read()
        result = parse_ls_lr(lines)
pprint(result)

In [ ]:
len(result)

In [ ]:
def filtrer_anciens(donnees, minutes=15) ->dict :
    maintenant = datetime.now()
    seuil = timedelta(minutes=minutes)

    resultats = {}

    # for rep, date in donnees.items():
    for rep in sorted(donnees):
        date = donnees[rep]
        if date is None:
            continue

        age = maintenant - date

        if age > seuil:
            resultats[rep] = {
                "date": date,
                "age_minutes": int(age.total_seconds() // 60)
            }

    return resultats

In [ ]:
def dict_to_html(data, minutes):
    html = []
    html.append(f"<H1>Synthèse pour {minutes} minutes à {datetime.now()}</H1>")
    html.append("<table border='1' cellpadding='5' cellspacing='0'>")
    html.append("<tr><th>Répertoire</th><th>Mise à jour</th><th>Âge (minutes)</th></tr>")

    for rep, info in data.items():
        date = info["date"]
        age = info["age_minutes"]

        # couleur si fichier trop ancien
        couleur = "#ffcccc" if age > minutes else "#ccffcc"

        html.append(
            f"<tr style='background-color:{couleur};'>"
            f"<td>{rep}</td>"
            f"<td>{date.strftime('%d/%m/%Y %H:%M')}</td>"
            f"<td>{age}</td>"
            f"</tr>"
        )

    html.append("</table>")

    return "\n".join(html)

In [ ]:
old_lines = filtrer_anciens(result, minutes = 120)

In [ ]:
HTML(dict_to_html(old_lines, minutes = 120))

In [ ]:
def write_html_to_file(html, file):
    with open(file, "w", encoding="utf-8") as f:
        f.write(html)

In [ ]:
def main_trt(txt_file_path, html_file_path = None, minutes=45):
    """
    :param txt_file_path: le path vers le fichier txt d'entrée
    :param html_file_path: le path vers le fihcier HTML a créer
    :param minute: la seuil de filtrage en minutes
    :return:
    """
    with open(txt_file_path, "r", encoding="utf-8") as f:
        lines = f.read()
        result_dict = parse_ls_lr(lines)
        old_lines = filtrer_anciens(result_dict, minutes)
        html = dict_to_html(old_lines,  minutes = minutes)
        if html_file_path: 
            write_html_to_file(html, html_file_path)
        else:
            return HTML(html)

In [ ]:
main_trt(Path("//sn1314/mips/tempo_bma/svc_logs.txt"),Path("//sn1314/mips/tempo_bma/svc_synthesis.html"),minutes = 45)

In [ ]:
main_trt(Path("//sn1314/mips/tempo_bma/trl_logs.txt"),Path("//sn1314/mips/tempo_bma/trl_synthesis.html"),minutes = 45)

In [ ]:
main_trt(Path("//sn1314/mips/tempo_bma/svc_logs.txt"))

In [ ]:
main_trt(Path("//sn1314/mips/tempo_bma/trl_logs.txt"))

In [ ]:
# Surveiller le départ de MsSanté

In [ ]:
from pathlib import Path
import os

In [ ]:
CIBLE = Path("//sn1314/mips/hprim_v3")

In [ ]:
dir(CIBLE)

In [ ]:
pdf_lst = CIBLE.glob("*.pdf")

In [ ]:
CIBLE.stat()

In [ ]:
from pathlib import Path
import time



In [ ]:
minutes = 10

In [ ]:
def panne_ms_sante(path: Path, minutes: int):
    """Vérifie qu'il n'y a pas de fichier de plus de NN minutes dans le repertoire de départ de MsSante"""

    rep = Path(path)
    now = time.time()     # Un float de type 1774448404.9379885
    seuil = minutes * 60  # en secondes

    fichiers = [
        f for f in rep.iterdir()
        if f.is_file()
        and f.suffix.lower() == '.pdf'
        and (now - f.stat().st_mtime > seuil)
    ]
    return len(fichiers)

In [ ]:
if attente := panne_ms_sante(CIBLE, minutes):
    print(f"En attente : {attente} fichiers")
else:
    print("MsSanté OK")


In [ ]:
attente

# Essayer de filtrer selon un ordre de liste

In [ ]:
txt_file_path = Path("//sn1314/mips/tempo_bma/svc_logs.txt")
with open(txt_file_path, "r", encoding="utf-8") as f:
    lines = f.read()
    result_dict = parse_ls_lr(lines)
    old_lines = filtrer_anciens(result_dict, minutes)
    html = dict_to_html(old_lines,  minutes = minutes)

In [ ]:
old_lines

In [ ]:
ordre = [f"glimscron{i}" for i in range(1, 25)] + [f"glimsonl{i}" for i in range(1, 25)] + ["JCE1", "JCE2", "JCE3"] 

In [ ]:
dict_in_ordered_list = {k : old_lines[k] for k in ordre if k in old_lines}

In [ ]:
{k : old_lines[k] for k in old_lines if k not in dict_in_ordered_list}

In [ ]:
ligne = "-rw-rw-r--. 1 clinisys users  5690941  1 avril 11:31 glims_hexa_ident_20260401_085455.log"

In [ ]:
match = re.search(r'(\d{1,2}) (\w+) +(\d{1,2}:\d{2})', ligne)

In [ ]:
match

In [ ]:
match.groups()